# P07: Applications and advanced tips

`pandas` is commonly used for behavioral and neural data analysis. We will build a
trial-level dataset, summarize it by condition, and then introduce a few pandas
pitfalls that can produce confident, error-free, but *wrong* results:
the mean-of-means problem, silent `NaN` dropping, and chained assignment.

## A simulated trial-level dataset

In [1]:
import numpy as np        # we'll cover numpy in detail in chapter 09
import pandas as pd

# just using numpy to get a random number generator
rng = np.random.default_rng(0)

# simulate a 2-condition RT experiment with unequal trials per subject
rows = []
for subj in range(1, 6):
    n_trials = rng.integers(8, 15)
    for _ in range(n_trials):
        cond = rng.choice(["congruent", "incongruent"])
        base = 500 if cond == "congruent" else 580
        rt = base + rng.normal(0, 60)
        rows.append((subj, cond, rt))

df = pd.DataFrame(rows, columns=["subject", "condition", "rt"])
df.head()

,subject,condition,rt
0,1,incongruent,572.073708
1,1,congruent,506.294007
2,1,congruent,467.859838
3,1,incongruent,658.240003
4,1,incongruent,636.824858


## Mean RT per condition -- the obvious way

In [2]:
# this pools all trials together, ignoring which subject they came from
df.groupby("condition")["rt"].mean()

condition
congruent      506.240035
incongruent    579.778684
Name: rt, dtype: float64

## Why that can be the wrong number

In [4]:
# wrong for a within-subject design, and subjects with more trials count more
naive = df.groupby("condition")["rt"].mean()

# correct: average within each subject first, then across subjects
# this will make a new df 'per_sub' with appropriate column names ('.reset_index()')
per_subj = df.groupby(["subject", "condition"])["rt"].mean().reset_index()
proper = per_subj.groupby("condition")["rt"].mean()

print("naive (pool all trials):\n", naive, "\n")
print("proper (subject means):\n", proper)

naive (pool all trials):
 condition
congruent      506.240035
incongruent    579.778684
Name: rt, dtype: float64 

proper (subject means):
 condition
congruent      502.716477
incongruent    584.233824
Name: rt, dtype: float64


:::{admonition} Advanced tip: mean-of-means vs grand mean
:class: tip
With unequal trial counts, pooling every trial weights heavier-sampled subjects
more, which is usually *not* what a within-subject design intends. Both numbers
compute without error and only your knowledge of the design tells you which is
correct. This is a common silent error when people start doing analyses,
and be careful to give concrete instructions to AI assistants.
:::

## Missing data: `NaN` is skipped, not flagged

In [5]:
# mark a few trials as missing (e.g., no response recorded)
df.loc[df.sample(5, random_state=1).index, "rt"] = np.nan

print("mean (NaN skipped):\n", df.groupby("condition")["rt"].mean(), "\n")
print("row count (NaN included):\n", df.groupby("condition")["rt"].size())

mean (NaN skipped):
 condition
congruent      506.604868
incongruent    575.570226
Name: rt, dtype: float64 

row count (NaN included):
 condition
congruent      26
incongruent    29
Name: rt, dtype: int64


:::{admonition} Advanced tip: `.mean()` silently changes your denominator
:class: tip
By default pandas skips `NaN`, so `.mean()` divides by the number of *present*
values, not your trial count. If those missing trials are not missing at random
(say, the hardest trials are the ones with no response), the result is biased with no error and no warning. Always inspect missingness (`df.isna().sum()`) and
decide how to explicitly handle it before you aggregate.
:::

:::{admonition} Advanced tips
:class: tip
* pandas 2.x is moving toward **copy-on-write**, which changes when a slice is a
  view vs a copy. Avoid *chained* indexing like `df[df.rt > 500]["rt"] = 0`
  (it may set a value on a throwaway copy and do nothing). Use a single `.loc`:
  `df.loc[df.rt > 500, "rt"] = 0`.
:::